# NPUFlow training agent

This notebook turns a Kaggle GPU session into a ClearML worker: it takes training
tasks from the ClearML queue `kaggle`, runs them, and leaves when the queue is empty
or the session time is nearly over.

**Before the first run** add two secrets (menu *Add-ons → Secrets*) and attach them to this notebook:

| Label | Value |
|---|---|
| `CLEARML_API_ACCESS_KEY` | access key from ClearML: *Settings → Workspace → Create new credentials* |
| `CLEARML_API_SECRET_KEY` | secret key from the same dialog |

Use a separate credentials pair for Kaggle so that it can be revoked on its own.

In [ ]:
QUEUE = "kaggle"       # ClearML queue to serve
MAX_HOURS = 11.3       # leave before Kaggle's 12-hour limit
IDLE_MINUTES = 10      # leave if there has been nothing to do for this long

In [ ]:
import os, sys, time, glob, socket, subprocess

START = time.time()
missing = []
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            missing.append(_name)
except Exception as e:
    missing = ["CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"]
    print("kaggle_secrets is not available:", e)

READY = not missing
if READY:
    os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
    os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml"
    os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"
    print("ClearML secrets found.")
else:
    print("Secrets missing or not attached to this notebook:", missing)
    print("Add them under Add-ons -> Secrets, tick the checkbox to attach them, then run again.")

In [ ]:
if READY:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "clearml", "clearml-agent"], check=True)
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout or "no GPU")
    print("CPU cores:", os.cpu_count())

In [ ]:
def find_dir(pattern):
    """Directory of the first file matching the pattern under /kaggle/input."""
    hits = glob.glob("/kaggle/input/**/" + pattern, recursive=True)
    return os.path.dirname(sorted(hits)[0]) if hits else None

if READY:
    chairs = find_dir("00001_img1.ppm")
    sintel = find_dir("training/clean/alley_1/frame_0001.png")
    kitti = find_dir("training/flow_occ/000000_10.png")
    if chairs:
        os.environ["NPUFLOW_CHAIRS"] = chairs
    if sintel:
        os.environ["NPUFLOW_SINTEL"] = sintel[: sintel.index("/training/clean")]
    if kitti:
        os.environ["NPUFLOW_KITTI"] = kitti[: kitti.index("/training/flow_occ")]
    for key in ("NPUFLOW_CHAIRS", "NPUFLOW_SINTEL", "NPUFLOW_KITTI", "NPUFLOW_THINGS"):
        print(key, "=", os.environ.get(key, "NOT FOUND"))

In [ ]:
if READY:
    from clearml.backend_api import Session

    session = Session()
    host = socket.gethostname()

    def call(service, action, **payload):
        return session.send_request(service=service, action=action, json=payload).json()["data"]

    queue_id = next(q["id"] for q in call("queues", "get_all", name=QUEUE)["queues"] if q["name"] == QUEUE)

    def queue_length():
        return len(call("queues", "get_by_id", queue=queue_id)["queue"].get("entries", []))

    def busy():
        return any(w["id"].startswith(host) and w.get("task") for w in call("workers", "get_all")["workers"])

    env = dict(os.environ)
    env["CLEARML_AGENT_SKIP_PIP_VENV_INSTALL"] = sys.executable   # use Kaggle's Python with its torch
    env["CLEARML_AGENT_SKIP_PYTHON_ENV_INSTALL"] = "1"
    env["NPUFLOW_DEADLINE"] = str(START + (MAX_HOURS - 0.25) * 3600)  # train.py saves and stops by then

    print(f"queue '{QUEUE}': {queue_length()} task(s) waiting")
    agent = subprocess.Popen(["clearml-agent", "daemon", "--queue", QUEUE, "--foreground"], env=env)
    idle_since = time.time()
    try:
        while agent.poll() is None:
            time.sleep(60)
            hours = (time.time() - START) / 3600
            if hours >= MAX_HOURS:
                print(f"{hours:.1f} h: session time is over, stopping the agent")
                break
            try:
                active = busy() or queue_length() > 0
            except Exception as e:      # a network hiccup must not kill a running training
                print("status check failed:", e)
                active = True
            if active:
                idle_since = time.time()
            elif time.time() - idle_since > IDLE_MINUTES * 60:
                print(f"nothing to do for {IDLE_MINUTES} min, stopping the agent")
                break
    finally:
        if agent.poll() is None:
            agent.terminate()
            try:
                agent.wait(timeout=120)
            except subprocess.TimeoutExpired:
                agent.kill()
    print(f"agent finished after {(time.time() - START) / 3600:.2f} h")